# Pathway ↔ Pathway Relation-Wise Merge

Merges Pathway–Pathway triples from PrimeKG and TARKG;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'


OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PATHWAY_PATHWAY/ALL_PATHWAY_PATHWAY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. PrimeKG

In [4]:
PrimeKG_Pathway_Pathway = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Pathway_Pathway.csv')
PrimeKG_Pathway_Pathway.columns = PrimeKG_Pathway_Pathway.columns.str.lower()
print(f"PrimeKG: {len(PrimeKG_Pathway_Pathway):,} rows | columns: {list(PrimeKG_Pathway_Pathway.columns)}")
PrimeKG_Pathway_Pathway['kg_type'] = 'Generalised'
PrimeKG_Pathway_Pathway['species'] = 'Homo species'

PrimeKG_Pathway_Pathway.head(2)

PrimeKG: 2,523 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'head_detail_name.1', 'tail_detail_name', 'tail_detail_name.1']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,head_detail_name.1,tail_detail_name,tail_detail_name.1,kg_type,species
0,R-HSA-109606,Pathway_Pathway,R-HSA-109581,Pathway,parent-child,Pathway,PrimeKG,Reactome_Pathway_ID,Reactome_Pathway_ID,Intrinsic Pathway for Apoptosis,Intrinsic Pathway for Apoptosis,Apoptosis,Apoptosis,Generalised,Homo species
1,R-HSA-169911,Pathway_Pathway,R-HSA-109581,Pathway,parent-child,Pathway,PrimeKG,Reactome_Pathway_ID,Reactome_Pathway_ID,Regulation of Apoptosis,Regulation of Apoptosis,Apoptosis,Apoptosis,Generalised,Homo species


### 1b. TARKG

In [5]:
TARKG_Pathway_Pathway = pd.read_csv(PROC_DIR + 'TARKG/Pathway_Pathway_TARKG.csv')
TARKG_Pathway_Pathway.columns = TARKG_Pathway_Pathway.columns.str.lower()
print(f"TARKG:   {len(TARKG_Pathway_Pathway):,} rows | columns: {list(TARKG_Pathway_Pathway.columns)}")

TARKG_Pathway_Pathway['kg_type'] = 'Generalised'
TARKG_Pathway_Pathway['kg_source'] = 'TARKG'
TARKG_Pathway_Pathway['species'] = 'Homo species'

TARKG_Pathway_Pathway.head(2)

TARKG:   2,431 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_index', 'kg', 'change']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,R-HSA-1500620,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Meiosis,Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2215978,BioKG,0,Generalised,Homo species
1,R-HSA-73886,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Chromosome Maintenance,Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2216900,BioKG,0,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [6]:
SOURCE_DFS = [
    ('PrimeKG_Pathway_Pathway', PrimeKG_Pathway_Pathway),
    ('TARKG_Pathway_Pathway',   TARKG_Pathway_Pathway),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[PrimeKG_Pathway_Pathway] ✓ no duplicates
[TARKG_Pathway_Pathway] ✓ no duplicates


In [7]:
PrimeKG_Pathway_Pathway = PrimeKG_Pathway_Pathway.loc[:, ~PrimeKG_Pathway_Pathway.columns.duplicated(keep='first')]
TARKG_Pathway_Pathway   = TARKG_Pathway_Pathway.loc[:,   ~TARKG_Pathway_Pathway.columns.duplicated(keep='first')]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[PrimeKG_Pathway_Pathway] ✓ clean
[TARKG_Pathway_Pathway] ✓ clean


## 3. Align Schemas and Concatenate

In [8]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 4,954 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,R-HSA-109606,Pathway_Pathway,R-HSA-109581,Pathway,parent-child,Pathway,PrimeKG,Reactome_Pathway_ID,Reactome_Pathway_ID,Intrinsic Pathway for Apoptosis,Apoptosis,Generalised,Homo species
1,R-HSA-169911,Pathway_Pathway,R-HSA-109581,Pathway,parent-child,Pathway,PrimeKG,Reactome_Pathway_ID,Reactome_Pathway_ID,Regulation of Apoptosis,Apoptosis,Generalised,Homo species


## 4. Standardise Fixed-Value Columns

In [9]:
final_df['head_type']  = 'Pathway'
final_df['tail_type']  = 'Pathway'
final_df['relation']   = 'Pathway_Pathway'
final_df['head_id_is'] = 'Reactome_ID'
final_df['tail_id_is'] = 'Reactome_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Pathway_Pathway'}
Unique kg_source: {'TARKG', 'PrimeKG'}


## 5. Deduplicate and Add Schema Columns

In [10]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 2,545 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,R-HSA-1059683,Pathway_Pathway,R-HSA-6783589,Pathway,parent-child,Pathway,PrimeKG::TARKG,Reactome_ID,Reactome_ID,Interleukin-6 signaling,Interleukin-6 family signaling,Generalised,Homo species
1,R-HSA-109581,Pathway_Pathway,R-HSA-5357801,Pathway,parent-child,Pathway,PrimeKG::TARKG,Reactome_ID,Reactome_ID,Apoptosis,Programmed Cell Death,Generalised,Homo species
2,R-HSA-109606,Pathway_Pathway,R-HSA-109581,Pathway,parent-child,Pathway,PrimeKG::TARKG,Reactome_ID,Reactome_ID,Intrinsic Pathway for Apoptosis,Apoptosis,Generalised,Homo species
3,R-HSA-109703,Pathway_Pathway,R-HSA-109704,Pathway,parent-child,Pathway,PrimeKG::TARKG,Reactome_ID,Reactome_ID,PKB-mediated events,PI3K Cascade,Generalised,Homo species
4,R-HSA-109704,Pathway_Pathway,R-HSA-112399,Pathway,parent-child,Pathway,PrimeKG::TARKG,Reactome_ID,Reactome_ID,PI3K Cascade,IRS-mediated signalling,Generalised,Homo species


## 6. QC — NaN Check

In [11]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 7. Save Output

In [13]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 2,545 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PATHWAY_PATHWAY/ALL_PATHWAY_PATHWAY.csv
